In [1]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import *

In [2]:
spark = (SparkSession
        .builder
        .appName("Spark User Defined Functions")
        .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/03 20:50:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.types import LongType

In [4]:
# Create cubed function
def cubed(s):
    return s * s * s

In [5]:
# Register UDF
spark.udf.register("cubed", cubed, LongType())

<function __main__.cubed(s)>

In [6]:
# Generate temporary view
spark.range(1, 9).createOrReplaceTempView("udf_test")

In [8]:
# Query the cubed UDF
spark.sql("SELECT id, cubed(id) AS id_cubed FROM udf_test").show()

[Stage 0:>                                                          (0 + 4) / 4]

+---+--------+
| id|id_cubed|
+---+--------+
|  1|       1|
|  2|       8|
|  3|      27|
|  4|      64|
|  5|     125|
|  6|     216|
|  7|     343|
|  8|     512|
+---+--------+



Spark SQL (this includes SQL, the DataFrame API, and the Dataset API) does not
guarantee the order of evaluation of subexpressions. For example, the following query
does not guarantee that the s is NOT NULL clause is executed prior to the strlen(s)> 1 clause:
```
spark.sql("SELECT s FROM test1 WHERE s IS NOT NULL AND strlen(s) > 1")
```
Therefore, to perform proper null checking, it is recommended that we do the
following:
1. Make the UDF itself null-aware and do null checking inside the UDF.
2. Use IF or CASE WHEN expressions to do the null check and invoke the UDF in a
conditional branch.


Previously, PySpark UDFs had slower performance than Scala UDFs because of data movement between Python and JVM. data movement between the JVM and Python, which was quite expensive. To resolve this problem, Pandas UDFs (also known as vectorized UDFs) were introduced. A Pandas UDF uses Apache Arrow to transfer data and Pandas
to work with the data. You define a Pandas UDF using the keyword pandas_udf as
the decorator, or to wrap the function itself. Once the data is in Apache Arrow for‐
mat, there is no longer the need to serialize/pickle the data as it is already in a format consumable by the Python process. Instead of operating on individual inputs row by row, you are operating on a Pandas Series or DataFrame (i.e., vectorized execution).


In [10]:
# Import pandas
import pandas as pd

In [11]:
# Import various pyspark SQL functions including pandas_udf
from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import LongType

In [12]:
# Declare the cubed function
def cubed(a: pd.Series) -> pd.Series:
    return a * a * a
    
# Create the pandas UDF for the cubed function
cubed_udf = pandas_udf(cubed, returnType=LongType())

The preceding code snippet declares a function called `cubed()` that performs a cubed
operation. This is a regular Pandas function with the additional `cubed_udf = pan
das_udf()` call to create our Pandas UDF.

In [14]:
# Create a Pandas Series
x = pd.Series([1, 2, 3])

# The function for a pandas_udf executed with local Pandas data
print(cubed(x))

0     1
1     8
2    27
dtype: int64


Now let’s switch to a Spark DataFrame. We can execute this function as a Spark vectorized UDF as follows:

In [15]:
# Create a Spark DataFrame, 'spark' is an existing SparkSession
df = spark.range(1, 4)

In [16]:
# Execute function as a Spark vectorized UDF
df.select("id", cubed_udf(col("id"))).show()

+---+---------+
| id|cubed(id)|
+---+---------+
|  1|        1|
|  2|        8|
|  3|       27|
+---+---------+



As opposed to a local function, using a vectorized UDF will result in the execution of
Spark jobs; the previous local function is a Pandas function executed only on the
Spark driver.

In [18]:
spark.stop()